In [29]:
import numpy as np
import pandas as pd

import time
from tqdm import tqdm
from tvDatafeed import TvDatafeed, Interval

def get_data_from_tradingview(
    tickers: list[str],
    interval: Interval, # tvDatafeed.Interval 객체 형태
    exchange: list[str],
    n_bars: int, # 충분히 큰 수. Tradingview는 기간이 아닌 candle bar의 개수로 호출 가능
    column: str | None = None,
    verbose: bool = True, # 종목을 얼마만큼 호출 성공했는지 progress bar로 표현
    num_trials: int = 5, # 호출 실패 시 재시도 횟수
    multi_level_index: bool = True, # yfinance에서의 attribute와 동일한 기능
    tz_cleansing: bool = False,
    session_duration_map: dict[str, tuple[int, int]] | None = None,
    duplicate_index_method: str = "last",  # "last", "first", "mean"
) -> pd.DataFrame:
    """
    TradingView(tvDatafeed)로부터 가격데이터 import

    Params
    - column:
        * str  : 해당 컬럼만 반환 (columns=tickers)
        * None : OHLCV 전체 반환

    - multi_level_index:
        * True  : columns=MultiIndex (ticker, field)  -> ("AAPL","close") 형태
        * False : columns=MultiIndex (field, ticker)  -> data["close"] 로 모든 종목 접근 가능

    - tz_cleansing:
        * True  : 인덱스를 날짜 단위로 정규화
        * False : 원본 인덱스 유지

    - session_duration_map:
        * None : 인덱스 변경 없음
        * dict : 거래소별 open -> close 시간 차이
            예:
            {
                "NASDAQ": (6, 30),
                "NYSE": (6, 30),
                "KRX": (6, 30),
                "TSE": (6, 0),
            }

    - duplicate_index_method:
        * "last"  : 같은 timestamp가 여러 개 있으면 마지막 값 사용
        * "first" : 같은 timestamp가 여러 개 있으면 첫 번째 값 사용
        * "mean"  : 같은 timestamp가 여러 개 있으면 평균값 사용

    Notes
    - column이 str이면 multi_level_index 설정은 의미가 거의 없고, 그냥 (columns=tickers)로 반환.
    """

    def _shift_index_by_session_duration(
        idx: pd.Index,
        exch: str,
        session_duration_map: dict[str, tuple[int, int]] | None,
    ) -> pd.DatetimeIndex:
        idx = pd.to_datetime(idx)

        if getattr(idx, "tz", None) is not None:
            idx = idx.tz_localize(None)

        if session_duration_map is None:
            return pd.DatetimeIndex(idx)

        if exch not in session_duration_map:
            return pd.DatetimeIndex(idx)

        hours, minutes = session_duration_map[exch]
        delta = pd.Timedelta(hours=hours, minutes=minutes)

        return pd.DatetimeIndex(idx + delta)

    def _remove_duplicate_index(obj: pd.Series | pd.DataFrame):
        """
        index가 중복된 Series/DataFrame을 정리.
        TradingView continuous futures, timezone cleansing, session shift 이후
        동일 날짜/시간 index가 생길 수 있으므로 concat 전에 반드시 처리.
        """
        if not obj.index.has_duplicates:
            return obj

        if duplicate_index_method == "last":
            return obj.groupby(level=0).last()

        elif duplicate_index_method == "first":
            return obj.groupby(level=0).first()

        elif duplicate_index_method == "mean":
            return obj.groupby(level=0).mean(numeric_only=True)

        else:
            raise ValueError(
                "duplicate_index_method must be one of ['last', 'first', 'mean']"
            )

    tv = TvDatafeed()
    iterator = tqdm(list(zip(tickers, exchange)), disable=not verbose)

    ohlcv_cols = ["open", "high", "low", "close", "volume"]
    frames: list[pd.DataFrame] = []
    series_list: list[pd.Series] = []

    for ticker, exch in iterator:
        got = False

        for attempt in range(num_trials):
            try:
                temp = tv.get_hist(
                    symbol=ticker,
                    exchange=exch,
                    interval=interval,
                    n_bars=n_bars,
                )

                if temp is None or temp.empty:
                    raise ValueError(f"Empty data returned for {ticker} ({exch}).")

                temp.columns = [c.lower() for c in temp.columns]
                temp.index = pd.to_datetime(temp.index)

                # 거래소별 open -> close shift
                if session_duration_map is not None:
                    temp.index = _shift_index_by_session_duration(
                        idx=temp.index,
                        exch=exch,
                        session_duration_map=session_duration_map,
                    )

                elif tz_cleansing:
                    temp.index = pd.to_datetime(temp.index.strftime("%Y-%m-%d"))

                else:
                    if getattr(temp.index, "tz", None) is not None:
                        temp.index = temp.index.tz_localize(None)

                # timezone cleansing, session shift, continuous futures rollover 등으로
                # 동일 timestamp가 생기는 경우를 여기서 제거
                temp = _remove_duplicate_index(temp)

                if column is None:
                    use_cols = [c for c in ohlcv_cols if c in temp.columns]
                    temp2 = temp[use_cols].copy()

                    # OHLCV DataFrame 기준으로도 한 번 더 방어
                    temp2 = _remove_duplicate_index(temp2)

                    temp2.columns = pd.MultiIndex.from_product(
                        [[ticker], temp2.columns.tolist()],
                        names=["ticker", "field"],
                    )
                    frames.append(temp2)

                else:
                    col = column.lower()
                    if col not in temp.columns:
                        raise KeyError(
                            f"Column '{column}' not in data columns for {ticker}: {list(temp.columns)}"
                        )

                    series = temp[col].copy()
                    series.name = ticker

                    # pd.concat(series_list, axis=1)에서 터지는 것을 방지
                    series = _remove_duplicate_index(series)

                    series_list.append(series)

                got = True
                break

            except Exception as e:
                if attempt < num_trials - 1:
                    time.sleep(1)
                else:
                    print(f"[FAIL] {ticker} ({exch}) after {num_trials} trials: {e}")

        if not got:
            continue

    if column is not None:
        if not series_list:
            return pd.DataFrame()

        # concat 직전 최종 방어
        series_list = [_remove_duplicate_index(s) for s in series_list]

        return pd.concat(series_list, axis=1).sort_index()

    if not frames:
        return pd.DataFrame()

    # OHLCV 전체 반환 시 concat 직전 최종 방어
    frames = [_remove_duplicate_index(f) for f in frames]

    out = pd.concat(frames, axis=1).sort_index()

    if multi_level_index:
        return out.sort_index(axis=1)

    out.columns = out.columns.swaplevel(0, 1)
    return out.sort_index(axis=1)

In [45]:
asset_data = {
    'KOSPI' : 'KRX', # KOSPI Composite
    'KOSPI200' : 'KRX', # KOSPI200
    'KOSDAQ' : 'KRX', # KOSDAQ
    'KOSDAQ150' : 'KRX',
    'SPX' : 'TVC', # S&P500
    'IXIC' : 'NASDAQ', # NASDAQ Composite
    'NDQ' : 'TVC', # NASDAQ100
    'DJI' : 'DJCFD', # Dow Jones
    'SX5E': 'TVC', # STOXX50
    'NIKKEI225' : 'Nikkei', # NIKKEI225
    '000001' : 'SSE' # SSEC
}

In [47]:
import yfinance as yf

asset_data_yf = yf.download(
    [
        '^GSPC','^IXIC','^DJI','^N225'
    ],
    start = '1990-01-01',
    progress = False,
    auto_adjust=False,
    multi_level_index=False
)

In [50]:
asset_data_yf['Close'].to_excel('yfdata.xlsx')

In [46]:
data = get_data_from_tradingview(
    tickers = list(asset_data.keys()),
    interval = Interval.in_daily,
    exchange = list(asset_data.values()),
    n_bars = 13000,
    column = 'close',
    verbose = True,
    num_trials = 5,
    multi_level_index = False,
    tz_cleansing = True
)

 82%|████████▏ | 9/11 [00:28<00:06,  3.37s/it]ERROR:tvDatafeed.main:Connection timed out
ERROR:tvDatafeed.main:no data, please check the exchange and symbol
ERROR:tvDatafeed.main:Connection timed out
ERROR:tvDatafeed.main:no data, please check the exchange and symbol
ERROR:tvDatafeed.main:Connection timed out
ERROR:tvDatafeed.main:no data, please check the exchange and symbol
ERROR:tvDatafeed.main:Connection timed out
ERROR:tvDatafeed.main:no data, please check the exchange and symbol
ERROR:tvDatafeed.main:Connection timed out
ERROR:tvDatafeed.main:no data, please check the exchange and symbol
 91%|█████████ | 10/11 [01:03<00:13, 13.11s/it]

[FAIL] NIKKEI225 (Nikkei) after 5 trials: Empty data returned for NIKKEI225 (Nikkei).


ERROR:tvDatafeed.main:Connection to remote host was lost.
ERROR:tvDatafeed.main:no data, please check the exchange and symbol
100%|██████████| 11/11 [01:06<00:00,  6.09s/it]
C:\Users\krm\AppData\Local\Temp\ipykernel_17168\1355288727.py:143: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  return pd.concat(series_list, axis=1).sort_index()


In [40]:
data

,KOSPI,KOSPI200,KOSDAQ,KOSDAQ150,SPX,IXIC,NDQ,DJI,SX5E,NI225,000001
datetime,,,,,,,,,,,
1973-09-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4587.40,NaN
1973-09-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4646.37,NaN
1973-09-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4620.91,NaN
1973-09-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4644.12,NaN
1973-09-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4623.79,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2026-05-20,7208.95,1125.51,1056.07,1773.46,7432.96,26270.3593,29297.6982,50013.9909,5976.08,59804.19,4162.1845
2026-05-21,7815.59,1225.22,1105.97,1875.52,7445.73,26293.0975,29357.2721,50291.0186,5960.33,61683.92,4077.2765
2026-05-22,7847.71,1226.03,1161.13,1991.85,7473.48,26343.9704,29481.6417,50585.0680,6019.46,63338.85,4112.8996


In [41]:
data.columns = [
    'KOSPI','KOSPI200','KOSDAQ','KOSDAQ150','SNP500','NASDAQ','NASDAQ100','DOW_JONES','EURO_STOXX_50','NIKKEI_225','SHANGHAI_COMP'
]

In [42]:
out_data = data.copy('deep')

In [43]:
out_data.index = out_data.index.strftime('%Y-%m-%d')

In [44]:
out_data.to_excel('indexes.xlsx')

In [17]:
data_long = data.loc['2020':].unstack().reset_index()

In [18]:
data_long.columns = ['INDC_CD','BASE_YMD','INDX_VAL']

In [20]:
data_long.dropna()

,INDC_CD,BASE_YMD,INDX_VAL
0,KOSPI,2020-01-02,2175.1699
1,KOSPI,2020-01-03,2176.4600
2,KOSPI,2020-01-06,2155.0701
3,KOSPI,2020-01-07,2175.5400
4,KOSPI,2020-01-08,2151.3101
...,...,...,...
16625,SHANGHAI_COMP,2026-05-18,4131.5276
16626,SHANGHAI_COMP,2026-05-19,4169.5378
16627,SHANGHAI_COMP,2026-05-20,4162.1845
16628,SHANGHAI_COMP,2026-05-21,4077.2765


### USBONDS

In [4]:
bonds_tickers = {
    'DGS2' : 'FRED',
    'DGS10' : 'FRED',
    'DGS30' : 'FRED'
}

In [5]:
bonds_data = get_data_from_tradingview(
    tickers = list(bonds_tickers.keys()),
    interval = Interval.in_daily,
    exchange = list(bonds_tickers.values()),
    n_bars = 10000,
    column = 'close',
    verbose = True,
    num_trials = 5,
    multi_level_index = False,
    tz_cleansing = True
)

100%|██████████| 3/3 [00:03<00:00,  1.10s/it]


In [6]:
bonds_data.index = bonds_data.index.strftime('%Y-%m-%d')

In [7]:
bonds_data.to_excel('bonds_us_fred.xlsx')

### Dollar Index

In [26]:
currency_tickers = {
    'DXY' : 'TVC',
    'JXY' : 'TVC',
    'EXY' : 'TVC',
    'BXY' : 'TVC',
    'SXY' : 'TVC',
    'CXY' : 'TVC',
    'AXY' : 'TVC',
    'ZXY' : 'TVC'
}

In [27]:
currency_data = get_data_from_tradingview(
    tickers = list(currency_tickers.keys()),
    interval = Interval.in_daily,
    exchange = list(currency_tickers.values()),
    n_bars = 10000,
    column = 'close',
    verbose = True,
    num_trials = 5,
    multi_level_index = False,
    tz_cleansing = True
)

100%|██████████| 8/8 [00:17<00:00,  2.14s/it]
C:\Users\krm\AppData\Local\Temp\ipykernel_17168\1355288727.py:143: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  return pd.concat(series_list, axis=1).sort_index()


In [29]:
currency_data.index = currency_data.index.strftime('%Y-%m-%d')

In [30]:
currency_data.to_excel('currency_data.xlsx')

### Commodity Futures

In [32]:
commodity_tickers = {
    'GC1!' : 'COMEX', # Gold
    'SI1!' : 'COMEX', # Silver
    'HG1!' : 'COMEX', # Copper
    'CL1!' : 'NYMEX', # Crude Oil
    'NG1!' : 'NYMEX', # Natural Gas
    'ZW1!' : 'CBOT', # Wheat
    'ZC1!' : 'CBOT', # Corn
}

In [33]:
commodity_data = get_data_from_tradingview(
    tickers = list(commodity_tickers.keys()),
    interval = Interval.in_daily,
    exchange = list(commodity_tickers.values()),
    n_bars = 10000,
    column = 'close',
    verbose = True,
    num_trials = 5,
    multi_level_index = False,
    tz_cleansing = True
)

  0%|          | 0/7 [00:00<?, ?it/s]ERROR:tvDatafeed.main:Connection to remote host was lost.
ERROR:tvDatafeed.main:no data, please check the exchange and symbol
100%|██████████| 7/7 [00:20<00:00,  2.94s/it]
C:\Users\krm\AppData\Local\Temp\ipykernel_17168\1355288727.py:143: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  return pd.concat(series_list, axis=1).sort_index()


In [36]:
commodity_data.index = commodity_data.index.strftime('%Y-%m-%d')

In [37]:
commodity_data.to_excel('commodity_data.xlsx')

In [30]:
commodity_symbols = [
    "ZW1!",
    "ZC1!",
    "ZS1!",
    "ZM1!",
    "ZL1!",
    "ZR1!",
    "ZO1!",
    "BARLEYJPR1!",
    "RS1!",
    #"RS1!",
    "FCPO1!",
    "SB1!",
    "W1!",
    "KC1!",
    "CC1!",
    "CT1!",
    "OJ1!",
    "TSR21!",
    "LBR1!",
    "LE1!",
    "GF1!",
    "HE1!",
    "DC1!",
    "BTR1!",
    "CSC1!",
    "CL1!",
    "BRN1!",
    "DBI1!",
    "NG1!",
    "RB1!",
    "HO1!",
    "G1!",
    "NCF1!",
    "UX1!",
    "EL1!",
    "GC1!",
    "SI1!",
    "PL1!",
    "PA1!",
    "HG1!",
    "ALI1!",
    #"ZINC1!",
    #"PB1!",
    #"NICKEL1!",
    #"SN1!",
    "COB1!",
    "LTH1!",
    "LTC1!",
    "TIO1!",
    #"SR1!",
    "HRC1!",
    #"CARBON1!",
]

In [31]:
exchanges = [
    "CBOT",
    "CBOT",
    "CBOT",
    "CBOT",
    "CBOT",
    "CBOT",
    "CBOT",
    "NCDEX",
    "ICEUS",
    #"ZCE",
    "MYX",
    "ICEUS",
    "ICEEUR",
    "ICEUS",
    "ICEUS",
    "ICEUS",
    "ICEUS",
    "TOCOM",
    "CME",
    "CME",
    "CME",
    "CME",
    "CME",
    "NZX",
    "CME",
    "NYMEX",
    "ICEEUR",
    "ICEEUR",
    "NYMEX",
    "NYMEX",
    "NYMEX",
    "ICEEUR",
    "ICEEUR",
    "COMEX",
    "NYMEX",
    "COMEX",
    "COMEX",
    "NYMEX",
    "NYMEX",
    "COMEX",
    "COMEX",
    #"MCX",
    #"LME",
    #"CAPITALCOM",
    #"LME",
    "COMEX",
    "COMEX",
    "COMEX",
    "COMEX",
    #"LME",
    "COMEX",
    #"CMC",
]

In [32]:
len(commodity_symbols)

45

In [37]:
commodity_data = get_data_from_tradingview(
    tickers = commodity_symbols,
    interval = Interval.in_daily,
    exchange = exchanges,
    n_bars = 13000,

    verbose = True,
    num_trials = 5,
    multi_level_index = True,
    tz_cleansing = True
)

  7%|▋         | 3/45 [00:10<02:31,  3.61s/it]ERROR:tvDatafeed.main:Connection to remote host was lost.
ERROR:tvDatafeed.main:no data, please check the exchange and symbol
 29%|██▉       | 13/45 [00:34<00:50,  1.59s/it]ERROR:tvDatafeed.main:Connection to remote host was lost.
ERROR:tvDatafeed.main:no data, please check the exchange and symbol
 33%|███▎      | 15/45 [00:38<00:53,  1.78s/it]ERROR:tvDatafeed.main:Connection to remote host was lost.
ERROR:tvDatafeed.main:no data, please check the exchange and symbol
100%|██████████| 45/45 [01:10<00:00,  1.56s/it]
C:\Users\Geust\AppData\Local\Temp\ipykernel_24284\882957812.py:200: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  out = pd.concat(frames, axis=1).sort_index()


In [44]:
with pd.ExcelWriter('commodity_futures_data.xlsx') as writer:
    for symbol in commodity_symbols:
        commodity_data[symbol].dropna(axis=0, how="all").to_excel(
            writer,
            sheet_name = symbol
        )

In [2]:
import numpy as np
import pandas as pd
import pandas_datareader as pdr

us02y = pdr.DataReader(
    'DGS02',
    data_source='FRED',
    start = '1990-01-01'
)

TypeError: deprecate_kwarg() missing 1 required positional argument: 'new_arg_name'